# 1. 模型与方程

## 1.1 从 Holmes 型 Duffing 振子到强耦合

原文式 (1) 是标准 Holmes 型：

$$
\ddot{x} + \xi\dot{x} - x + x^3 = F\cos t
$$

做时间尺度变换 $t=\omega\tau$（原文式 2），把固定频率 1 rad/s 换成可调的 $\omega$：

$$
\ddot{x} + \omega\xi\dot{x} - \omega^2(x-x^3) = \omega^2 F\cos(\omega t)
$$

这一步决定了后面所有参数的量纲：$\omega$ 越大，同一个 $F$ 作用越猛，步长必须跟着缩。

强耦合结构（原文式 4）是两个振子**同时耦合恢复力与阻尼力**：

$$
\begin{aligned}
\ddot{x}_1 + \omega\xi\dot{x}_1 + \omega^2\xi k(x_1-x_2) + \omega k(\dot{x}_1-\dot{x}_2) - \omega^2(x_1-x_1^3) &= \omega^2\big[F\cos(\omega t)+s(t)+n(t)\big]\\
\ddot{x}_2 + \omega\xi\dot{x}_2 + \omega^2\xi k(x_2-x_1) + \omega k(\dot{x}_2-\dot{x}_1) - \omega^2(x_2-x_2^3) &= \omega^2 F\cos(\omega t)
\end{aligned}
$$

| 符号 | 含义 | 表 1 默认值 | 取值区间 |
|---|---|---|---|
| $F$ | 策动力幅值 | 0.2 | [0, 2] |
| $\omega$ | 时间尺度因子 (rad/s) | $5\times10^6$ | $10^5 \sim 5\times10^7$ |
| $k$ | 耦合系数 | 10 | $10^{-1} \sim 10^3$ |
| $\xi$ | 阻尼系数 | 0.7 | $10^{-2} \sim 10^2$ |
| $h$ | 计算步长 | 1 ns | 20 ~ 1 ns |

初始值 $x_1=x_2=1.4$、$\dot x_1=\dot x_2=4$（原文 5.2 节）。


In [1]:
import sys, json, pathlib
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, "_src")
import c2sim as S

DATA = pathlib.Path("_data")
plt.rcParams.update({
    "figure.dpi": 110, "font.size": 10,
    "font.sans-serif": ["Microsoft YaHei", "SimHei", "DejaVu Sans"],
    "axes.unicode_minus": False,
})

def load(name):
    return json.loads((DATA / f"{name}.json").read_text(encoding="utf-8"))

def dual_plot(x, snri, r, xlabel, title="", logx=False, invert=False,
              marks=None, ymax=None):
    fig, ax1 = plt.subplots(figsize=(7.4, 3.5))
    ax1.plot(x, snri, "o-", color="#1f4e79", ms=4, lw=1.2)
    ax1.set_xlabel(xlabel); ax1.set_ylabel("SNRI / dB", color="#1f4e79")
    ax1.tick_params(axis="y", colors="#1f4e79")
    if ymax: ax1.set_ylim(top=ymax)
    ax2 = ax1.twinx()
    ax2.plot(x, r, "s--", color="#8a3b2e", ms=4, lw=1.2)
    ax2.set_ylabel("波形相似度 r", color="#8a3b2e")
    ax2.tick_params(axis="y", colors="#8a3b2e")
    if logx: ax1.set_xscale("log")
    if invert: ax1.invert_xaxis()
    if marks:
        for xv, lab in marks:
            ax1.axvline(xv, color="gray", ls=":", lw=1)
            ax1.text(xv, ax1.get_ylim()[1], lab, fontsize=8, va="top",
                     ha="center", color="#7a2e12")
    ax1.grid(alpha=.2); ax1.set_title(title)
    plt.tight_layout(); plt.show()


## 1.2 差模方程：强耦合到底做了什么

两个方程相减（令 $\Delta=x_1-x_2$，注意**不除以 2**）：

$$
\ddot\Delta+(\omega\xi+2\omega k)\dot\Delta+\omega^2(2\xi k-1)\Delta+\omega^2(x_1^3-x_2^3)=\omega^2\big[s(t)+n(t)\big]
$$

这一个式子就能解释原文第 4 节的两整节经验曲线：

- **刚度项的符号**由 $2\xi k-1$ 决定。$k<k_c=1/(2\xi)=0.714$ 时它是**负**的，
  差模仍是双阱——脉冲激励下两振子各奔一个阱，"差分"就等于把信号扔掉。
  这正是原文「$0<k<1$ 时检测失效」的数学原因。
- **阻尼项**多了 $2\omega k$。等效阻尼比
  $\zeta=(\xi+2k)/\big(2\sqrt{2\xi k-1}\big)$，$k=10$ 时等于 **2.87**，**过阻尼**。
  弱耦合（$k\to0$）时差模是欠阻尼双阱，会振铃；强耦合把它改成过阻尼单阱跟随。
  这才是"能恢复波形"的真正原因——**不是耦合变大，是差模换了一套阻尼**。


In [2]:
xi = 0.7
print(f"k_c = 1/(2*xi) = {1/(2*xi):.4f}   ← 差模由双阱变单阱的门限\n")
print(f"{'k':>7}{'2ξk−1':>9}{'ζ':>8}{'准静态增益':>12}  判定")
for k in (0.3, 0.5, 0.7, 0.714, 1, 3, 10, 100):
    d = 2*xi*k - 1
    if d <= 0:
        print(f"{k:>7.3f}{d:>9.3f}{'—':>8}{'—':>12}  负刚度：差模仍是双阱")
    else:
        z = (xi + 2*k) / (2*np.sqrt(d))
        print(f"{k:>7.3f}{d:>9.3f}{z:>8.2f}{1/d:>12.4f}  {'过阻尼' if z>1 else '欠阻尼'}")


k_c = 1/(2*xi) = 0.7143   ← 差模由双阱变单阱的门限

      k    2ξk−1       ζ       准静态增益  判定
  0.300   -0.580       —           —  负刚度：差模仍是双阱
  0.500   -0.300       —           —  负刚度：差模仍是双阱
  0.700   -0.020       —           —  负刚度：差模仍是双阱
  0.714   -0.000       —           —  负刚度：差模仍是双阱
  1.000    0.400    2.13      2.5000  过阻尼
  3.000    3.200    1.87      0.3125  过阻尼
 10.000   13.000    2.87      0.0769  过阻尼
100.000  139.000    8.51      0.0072  过阻尼


准静态下忽略非线性项，$\Delta \approx s/(2\xi k-1)$。$k=10$ 时增益 $1/13$。
**这个增益小于 1**，所以差分通道不会凭空提高灵敏度——信号和噪声被同比例压低，SNR 不变。
第 4 章会看到，这正是"必须靠限带来换信噪比"的原因。

实测增益比 $1/13$ 还低一些，因为 $x_1^3-x_2^3\approx3x^2\Delta$ 提供了一部分负刚度修正。
下面验证无信号时两振子是否严格同步。

In [3]:
T, h = 40e-6, 1e-9
t = np.arange(int(round(T/h)) + 1) * h
print(f"时长 {T*1e6:.0f} us，步长 {h*1e9:.0f} ns，共 {len(t)} 点，"
      f"约 {T/(2*np.pi/5e6):.1f} 个激励周期")

so0 = S.simulate(t, batch=1)
print(f"无信号时 max|ẋ1−ẋ2| = {np.abs(so0).max():.2e}   ← 浮点零，不是「很小」")
so1 = S.simulate(t, drive=lambda tt: 0.0*tt, batch=1)
print(f"零驱动同解：max|ẋ1−ẋ2| = {np.abs(so1).max():.2e}")


时长 40 us，步长 1 ns，共 40001 点，约 31.8 个激励周期


无信号时 max|ẋ1−ẋ2| = 1.16e-10   ← 浮点零，不是「很小」


零驱动同解：max|ẋ1−ẋ2| = 1.16e-10


结果是机器精度 0。这意味着**任何非零的 $\dot x_1-\dot x_2$ 都由 $s(t)$ 或 $n(t)$ 直接引起**，
没有基线偏置要标定——原文说它"不受初始状态限制"即指此。第 4 章会看到，这个性质让
$P_{fa}$ 可以解析算出。

下一章看没有信号时系统自身的相态地图。